# ARMADO DEL DATASET POST API

# DATOS

“El dataset base proviene de Make Music Equal (licencia CC BY-SA 4.0).
Este fue enriquecido con datos obtenidos mediante la API de Chartmetric, los cuales poseen restricciones de uso.
Por este motivo, las variables derivadas de dicha fuente fueron transformadas y agregadas de forma tal que no permiten reconstruir los valores originales.”

PASO 1: Base de artistas para obtener IDs antes de conectar a la API
Fuente: Make Music Equal (CC BY-SA 4.0)
La base de datos de artistas se descargó del dataset de acceso público MakeMusicEqual versión 3/3/2026: 936420, 10.
https://chartmetric-public.s3.us-west-2.amazonaws.com/make-music-equal/mme_artist_info.csv 
Se eliminaron registros duplicados [nombre + url de artista]: 172
Resultado: 936248 filas/artistas. 
Legal: This work is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License.


PASO 2: Construcción de las cohortes de datos a extraer
Base de datos MakeMusicEqual ordenada por Chartmetric_rank (rank 1 a 936248)
-cohorte 1: 1400 registros 
-cohorte 2: 5600 registros 

Notas:
Hay discontinuidades en el atributo Chartmetric_rank (faltan números en la secuencia) porque la base makemusicequal está actualizada al 3/3/26 aunque fueron descargados el 23/3/2026. Misma fecha de acceso a la API. Como la variable chartmetrick_rank es dinámica, esa diferencia de 20 días hace que un artista pueda tener rank = 5 el 3/3/2026 y rank = 9 el 23/3/26. Sin embargo, eso no invalida la selección del conjunto de artistas sobre la cual se construye la extracción de datos porque el objetivo fue definir un conjunto de artistas ordenados por popularidad (medida por una fuenta de datos confiable). Además la variable rank no será utilizada en el análisis. Y por último, se comprueba que el rank puede variar sutilmente pero el conjunto de artistas definidos es predominantemente coincidente. 

PASO 3: API. Extracción de datos
Límite 14 días, 100 request x día. (inicio 23/3/26)
Se apunta a dos endpoints: artist metadata y artist live events
La extracción de datos se fragmenta por cohortes y se va guardando en archivos parciales

POSTERIORMENTE se recuperó más info del jason:
Band
y
num_sp_editorial_playlists",
    "num_sp_playlists",
    "sp_editorial_playlist_total_reach",
    "num_am_editorial_playlists",
    "num_am_playlists",
    "num_de_editorial_playlists",
    "num_de_playlists",
    "de_playlist_total_reach",
    "de_editorial_playlist_total_reach",
    "num_az_editorial_playlists",
    "num_az_playlists",
    "num_yt_editorial_playlists",
    "num_yt_playlists",
    "yt_playlist_total_reach",
    "tiktok_top_video_views",
    "tiktok_top_video_comments",
    "tiktok_track_posts"


In [102]:
import pandas as pd
import numpy as np

# live events agregado

In [103]:
ruta_eventlevel = r"chartmetric_liveevents_2024_2025\liveevents_2024_2025_eventlevel.csv"

In [104]:
df_events = pd.read_csv(ruta_eventlevel)

In [105]:
print(df_events.shape)
print(df_events.columns.tolist())
df_events.head(3)

(35810, 24)
['downloaded_at_utc', 'chartmetric_id', 'artist_name', 'event_id', 'event_name', 'type', 'event_year', 'start_date', 'end_date', 'venue_id', 'venue_name', 'venue_capacity', 'city_id', 'city_name', 'code2', 'price', 'low_price', 'high_price', 'currency', 'is_headliner', 'jambase_event_url', 'songkick_event_url', 'seatgeek_event_url', 'ticketmaster_event_url']


,downloaded_at_utc,chartmetric_id,artist_name,event_id,event_name,type,event_year,start_date,end_date,venue_id,...,code2,price,low_price,high_price,currency,is_headliner,jambase_event_url,songkick_event_url,seatgeek_event_url,ticketmaster_event_url
0,2026-03-24T13:17:35.691409+00:00,214945,Bad Bunny,8674164,Bad Bunny at Estadio GNP Seguros,Concert,2025,2025-12-21 20:00:00.000,2025-12-21 23:59:59.000,1132.0,...,MX,986.0,4741.0,2000.0,USD,True,https://www.jambase.com/show/bad-bunny-estadio...,NaN,NaN,https://www.ticketmaster.com.mx/bad-bunny-debi...
1,2026-03-24T13:17:35.691421+00:00,214945,Bad Bunny,8674163,Bad Bunny at Estadio GNP Seguros,Concert,2025,2025-12-20 21:00:00.000,2025-12-20 23:59:59.000,1132.0,...,MX,4741.0,4741.0,2000.0,MXN,True,https://www.jambase.com/show/bad-bunny-estadio...,NaN,NaN,https://www.ticketmaster.com.mx/bad-bunny-debi...
2,2026-03-24T13:17:35.691428+00:00,214945,Bad Bunny,8674162,Bad Bunny at Estadio GNP Seguros,Concert,2025,2025-12-19 21:00:00.000,2025-12-19 23:59:59.000,1132.0,...,MX,986.0,NaN,NaN,USD,True,https://www.jambase.com/show/bad-bunny-estadio...,NaN,NaN,https://www.ticketmaster.com.mx/bad-bunny-debi...


In [106]:
# fechas
df_events["start_date"] = pd.to_datetime(df_events["start_date"], errors="coerce")
df_events["end_date"] = pd.to_datetime(df_events["end_date"], errors="coerce")

# por seguridad, si event_year vino mal tipado
df_events["event_year"] = pd.to_numeric(df_events["event_year"], errors="coerce")

# columnas numéricas que vamos a usar
cols_numericas = ["low_price", "high_price", "venue_capacity"]

for col in cols_numericas:
    df_events[col] = pd.to_numeric(df_events[col], errors="coerce")

In [107]:
print(df_events["event_year"].value_counts(dropna=False).sort_index())

event_year
2024    14373
2025    21437
Name: count, dtype: int64


In [108]:
def moneda_principal(serie):
    serie_limpia = serie.dropna()
    if serie_limpia.empty:
        return np.nan
    return serie_limpia.mode().iloc[0]

In [109]:
def resumir_live_por_anio(df, anio):
    df_anio = df[df["event_year"] == anio].copy()
    
    resumen = (
        df_anio
        .groupby(["chartmetric_id", "artist_name"], as_index=False)
        .agg(
            n_shows=("event_id", "nunique"),
            avg_low_price=("low_price", "mean"),
            avg_high_price=("high_price", "mean"),
            avg_venue_capacity=("venue_capacity", "mean"),
            total_capacity=("venue_capacity", "sum"),
            n_shows_with_capacity=("venue_capacity", lambda x: x.notna().sum()),
            n_cities=("city_name", lambda x: x.dropna().nunique()),
            n_countries=("code2", lambda x: x.dropna().nunique()),
            n_currencies=("currency", lambda x: x.dropna().nunique()),
            main_currency=("currency", moneda_principal)
        )
    )
    
    resumen = resumen.rename(columns={
        "n_shows": f"n_shows_{anio}",
        "avg_low_price": f"avg_low_price_{anio}",
        "avg_high_price": f"avg_high_price_{anio}",
        "avg_venue_capacity": f"avg_venue_capacity_{anio}",
        "total_capacity": f"total_capacity_{anio}",
        "n_shows_with_capacity": f"n_shows_with_capacity_{anio}",
        "n_cities": f"n_cities_{anio}",
        "n_countries": f"n_countries_{anio}",
        "n_currencies": f"n_currencies_{anio}",
        "main_currency": f"main_currency_{anio}"
    })
    
    # si no hay ningún dato de capacidad, dejar total_capacity como NaN
    col_total = f"total_capacity_{anio}"
    col_ncap = f"n_shows_with_capacity_{anio}"
    resumen.loc[resumen[col_ncap] == 0, col_total] = np.nan
    
    return resumen

In [110]:
df_live_2024 = resumir_live_por_anio(df_events, 2024)
df_live_2025 = resumir_live_por_anio(df_events, 2025)

print("df_live_2024:", df_live_2024.shape)
print("df_live_2025:", df_live_2025.shape)

df_live_2024: (816, 12)
df_live_2025: (941, 12)


In [111]:
df_live_agg = df_live_2024.merge(
    df_live_2025,
    on=["chartmetric_id", "artist_name"],
    how="outer"
)

In [112]:
cols_cero = [
    "n_shows_2024",
    "n_cities_2024",
    "n_countries_2024",
    "n_currencies_2024",
    "n_shows_with_capacity_2024",
    "n_shows_2025",
    "n_cities_2025",
    "n_countries_2025",
    "n_currencies_2025",
    "n_shows_with_capacity_2025"
]

for col in cols_cero:
    if col in df_live_agg.columns:
        df_live_agg[col] = df_live_agg[col].fillna(0)

In [113]:
df_live_agg["shows_per_country_2025"] = np.where(
    df_live_agg["n_countries_2025"] > 0,
    df_live_agg["n_shows_2025"] / df_live_agg["n_countries_2025"],
    np.nan
)

df_live_agg["shows_per_country_2024"] = np.where(
    df_live_agg["n_countries_2024"] > 0,
    df_live_agg["n_shows_2024"] / df_live_agg["n_countries_2024"],
    np.nan
)

## columnas bianuales

In [114]:
# columnas combinadas 2024 + 2025
df_live_agg["n_shows_total"] = (
    df_live_agg["n_shows_2024"] + df_live_agg["n_shows_2025"]
)

df_live_agg["n_shows_with_capacity_total"] = (
    df_live_agg["n_shows_with_capacity_2024"] + df_live_agg["n_shows_with_capacity_2025"]
)

df_live_agg["total_capacity_total"] = (
    df_live_agg["total_capacity_2024"].fillna(0) + df_live_agg["total_capacity_2025"].fillna(0)
)

# si no hubo ningún show con capacidad, dejar total_capacity_total como NaN
df_live_agg.loc[
    df_live_agg["n_shows_with_capacity_total"] == 0,
    "total_capacity_total"
] = np.nan

df_live_agg["avg_venue_capacity_total"] = np.where(
    df_live_agg["n_shows_with_capacity_total"] > 0,
    df_live_agg["total_capacity_total"] / df_live_agg["n_shows_with_capacity_total"],
    np.nan
)



In [115]:
def resumir_live_total(df):
    df_total = df[df["event_year"].isin([2024, 2025])].copy()
    
    resumen_total = (
        df_total
        .groupby(["chartmetric_id", "artist_name"], as_index=False)
        .agg(
            n_cities_total=("city_name", lambda x: x.dropna().nunique()),
            n_countries_total=("code2", lambda x: x.dropna().nunique())
        )
    )
    
    return resumen_total

In [116]:
df_live_total = resumir_live_total(df_events)

df_live_agg = df_live_agg.merge(
    df_live_total,
    on=["chartmetric_id", "artist_name"],
    how="left"
)

In [117]:
df_live_agg["shows_per_country_total"] = np.where(
    df_live_agg["n_countries_total"] > 0,
    df_live_agg["n_shows_total"] / df_live_agg["n_countries_total"],
    np.nan
)

In [118]:
print(df_live_agg.shape)
print(df_live_agg.columns.tolist())
df_live_agg.head(10)

(1004, 31)
['chartmetric_id', 'artist_name', 'n_shows_2024', 'avg_low_price_2024', 'avg_high_price_2024', 'avg_venue_capacity_2024', 'total_capacity_2024', 'n_shows_with_capacity_2024', 'n_cities_2024', 'n_countries_2024', 'n_currencies_2024', 'main_currency_2024', 'n_shows_2025', 'avg_low_price_2025', 'avg_high_price_2025', 'avg_venue_capacity_2025', 'total_capacity_2025', 'n_shows_with_capacity_2025', 'n_cities_2025', 'n_countries_2025', 'n_currencies_2025', 'main_currency_2025', 'shows_per_country_2025', 'shows_per_country_2024', 'n_shows_total', 'n_shows_with_capacity_total', 'total_capacity_total', 'avg_venue_capacity_total', 'n_cities_total', 'n_countries_total', 'shows_per_country_total']


,chartmetric_id,artist_name,n_shows_2024,avg_low_price_2024,avg_high_price_2024,avg_venue_capacity_2024,total_capacity_2024,n_shows_with_capacity_2024,n_cities_2024,n_countries_2024,...,main_currency_2025,shows_per_country_2025,shows_per_country_2024,n_shows_total,n_shows_with_capacity_total,total_capacity_total,avg_venue_capacity_total,n_cities_total,n_countries_total,shows_per_country_total
0,2,Massive Attack,11.0,54.666667,106.500000,7260.000000,36300.0,5.0,8.0,4.0,...,EUR,1.625000,2.750000,24.0,12.0,113693.0,9474.416667,20,11,2.181818
1,9,Stevie Wonder,13.0,92.833333,3469.166667,19118.230769,248537.0,13.0,12.0,1.0,...,USD,6.000000,13.000000,25.0,23.0,425142.0,18484.434783,19,2,12.500000
2,12,Rammstein,14.0,NaN,NaN,58531.875000,468255.0,8.0,4.0,3.0,...,NaN,NaN,4.666667,14.0,8.0,468255.0,58531.875000,4,3,4.666667
3,15,Alanis Morissette,30.0,70.269231,3631.307692,18819.933333,564598.0,30.0,27.0,2.0,...,USD,2.066667,15.000000,61.0,58.0,915016.0,15776.137931,48,16,3.812500
4,16,Beck,9.0,93.250000,1872.500000,10293.444444,92641.0,9.0,8.0,1.0,...,USD,4.750000,9.000000,28.0,27.0,211843.0,7846.037037,22,4,7.000000
5,22,Radiohead,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,...,NaN,4.000000,NaN,20.0,20.0,324000.0,16200.000000,5,5,4.000000
6,29,Metallica,21.0,79.861111,461.888889,60311.300000,1206226.0,20.0,11.0,6.0,...,USD,4.142857,3.500000,50.0,45.0,2764045.0,61423.222222,30,10,5.000000
7,33,Mariah Carey,35.0,162.012069,3915.968966,14573.714286,510080.0,35.0,23.0,2.0,...,USD,6.500000,17.500000,61.0,54.0,615120.0,11391.111111,26,5,12.200000
8,40,The Temptations,26.0,96.384615,811.000000,2844.739130,65429.0,23.0,24.0,2.0,...,USD,24.333333,13.000000,99.0,92.0,255483.0,2776.989130,83,3,33.000000
9,41,Annie Lennox,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,...,GBP,1.000000,NaN,2.0,2.0,7169.0,3584.500000,2,2,1.000000


In [119]:
print("Duplicados por chartmetric_id:", df_live_agg["chartmetric_id"].duplicated().sum())
print("Duplicados por artist_name:", df_live_agg["artist_name"].duplicated().sum())

Duplicados por chartmetric_id: 0
Duplicados por artist_name: 0


In [120]:
df_live_agg.isna().sum().sort_values(ascending=False)

avg_low_price_2024             414
avg_high_price_2024            414
main_currency_2024             360
avg_low_price_2025             314
avg_high_price_2025            314
main_currency_2025             280
total_capacity_2024            226
avg_venue_capacity_2024        226
shows_per_country_2024         194
avg_venue_capacity_2025        107
total_capacity_2025            107
shows_per_country_2025          73
total_capacity_total            29
avg_venue_capacity_total        29
shows_per_country_total          1
n_shows_2025                     0
n_cities_2024                    0
n_countries_2024                 0
n_currencies_2024                0
n_shows_with_capacity_2024       0
n_shows_2024                     0
chartmetric_id                   0
artist_name                      0
n_countries_2025                 0
n_currencies_2025                0
n_shows_with_capacity_2025       0
n_cities_2025                    0
n_shows_with_capacity_total      0
n_shows_total       

In [121]:
ruta_salida_live_agg = r"chartmetric_liveevents_2024_2025\liveevents_2024_2025_artist_agg.csv"
df_live_agg.to_csv(ruta_salida_live_agg, index=False)

In [122]:
df_live_agg.shape


(1004, 31)

In [123]:
df_live_agg.columns


Index(['chartmetric_id', 'artist_name', 'n_shows_2024', 'avg_low_price_2024',
       'avg_high_price_2024', 'avg_venue_capacity_2024', 'total_capacity_2024',
       'n_shows_with_capacity_2024', 'n_cities_2024', 'n_countries_2024',
       'n_currencies_2024', 'main_currency_2024', 'n_shows_2025',
       'avg_low_price_2025', 'avg_high_price_2025', 'avg_venue_capacity_2025',
       'total_capacity_2025', 'n_shows_with_capacity_2025', 'n_cities_2025',
       'n_countries_2025', 'n_currencies_2025', 'main_currency_2025',
       'shows_per_country_2025', 'shows_per_country_2024', 'n_shows_total',
       'n_shows_with_capacity_total', 'total_capacity_total',
       'avg_venue_capacity_total', 'n_cities_total', 'n_countries_total',
       'shows_per_country_total'],
      dtype='str')

In [124]:
df_live_agg.head()


,chartmetric_id,artist_name,n_shows_2024,avg_low_price_2024,avg_high_price_2024,avg_venue_capacity_2024,total_capacity_2024,n_shows_with_capacity_2024,n_cities_2024,n_countries_2024,...,main_currency_2025,shows_per_country_2025,shows_per_country_2024,n_shows_total,n_shows_with_capacity_total,total_capacity_total,avg_venue_capacity_total,n_cities_total,n_countries_total,shows_per_country_total
0,2,Massive Attack,11.0,54.666667,106.500000,7260.000000,36300.0,5.0,8.0,4.0,...,EUR,1.625000,2.750000,24.0,12.0,113693.0,9474.416667,20,11,2.181818
1,9,Stevie Wonder,13.0,92.833333,3469.166667,19118.230769,248537.0,13.0,12.0,1.0,...,USD,6.000000,13.000000,25.0,23.0,425142.0,18484.434783,19,2,12.500000
2,12,Rammstein,14.0,NaN,NaN,58531.875000,468255.0,8.0,4.0,3.0,...,NaN,NaN,4.666667,14.0,8.0,468255.0,58531.875000,4,3,4.666667
3,15,Alanis Morissette,30.0,70.269231,3631.307692,18819.933333,564598.0,30.0,27.0,2.0,...,USD,2.066667,15.000000,61.0,58.0,915016.0,15776.137931,48,16,3.812500
4,16,Beck,9.0,93.250000,1872.500000,10293.444444,92641.0,9.0,8.0,1.0,...,USD,4.750000,9.000000,28.0,27.0,211843.0,7846.037037,22,4,7.000000


In [125]:
df_live_agg.isna().sum().sort_values(ascending=False).head(15)

avg_low_price_2024          414
avg_high_price_2024         414
main_currency_2024          360
avg_low_price_2025          314
avg_high_price_2025         314
main_currency_2025          280
total_capacity_2024         226
avg_venue_capacity_2024     226
shows_per_country_2024      194
avg_venue_capacity_2025     107
total_capacity_2025         107
shows_per_country_2025       73
total_capacity_total         29
avg_venue_capacity_total     29
shows_per_country_total       1
dtype: int64

In [126]:
print("Artistas únicos en eventlevel:", df_events["chartmetric_id"].nunique())
print("Filas en df_live_agg:", df_live_agg.shape[0])

Artistas únicos en eventlevel: 1004
Filas en df_live_agg: 1004


In [127]:
df_live_summary = pd.read_csv(
    r"chartmetric_liveevents_2024_2025\liveevents_2024_2025_summary.csv"
)

In [128]:
print("Artistas en live summary:", df_live_summary["chartmetric_id"].nunique())

Artistas en live summary: 1400


In [129]:
ids_eventlevel = set(df_live_agg["chartmetric_id"].unique())
ids_summary = set(df_live_summary["chartmetric_id"].unique())

faltan_en_eventlevel = ids_summary - ids_eventlevel

print("Artistas en live_summary:", len(ids_summary))
print("Artistas en df_live_agg:", len(ids_eventlevel))
print("Artistas que están en summary pero no en eventlevel:", len(faltan_en_eventlevel))

Artistas en live_summary: 1400
Artistas en df_live_agg: 1004
Artistas que están en summary pero no en eventlevel: 396


In [130]:
df_faltantes = df_live_summary[
    df_live_summary["chartmetric_id"].isin(faltan_en_eventlevel)
].copy()

print(df_faltantes.shape)
df_faltantes.head(10)

(396, 13)


,downloaded_at_utc,chartmetric_id,artist_name,ok,status_code,error_text,raw_json_path,n_shows_2024,n_shows_2025,n_requests_live_2024_2025,x_ratelimit_limit,x_ratelimit_remaining,x_ratelimit_reset
3,2026-03-24T13:17:46.436509+00:00,3479,Justin Bieber,True,200,NaN,chartmetric_liveevents_2024_2025\raw_json\arti...,0,0,1,1000,964,1774424569
6,2026-03-24T13:21:31.117207+00:00,2316,Rihanna,True,200,NaN,chartmetric_liveevents_2024_2025\raw_json\arti...,0,0,1,1000,961,1774424569
10,2026-03-24T13:21:47.708582+00:00,3963,Ariana Grande,True,200,NaN,chartmetric_liveevents_2024_2025\raw_json\arti...,0,0,1,1000,956,1774424569
26,2026-03-24T13:22:34.009124+00:00,178,Michael Jackson,True,200,NaN,chartmetric_liveevents_2024_2025\raw_json\arti...,0,0,1,1000,939,1774424569
29,2026-03-24T13:23:06.983340+00:00,3168,Selena Gomez,True,200,NaN,chartmetric_liveevents_2024_2025\raw_json\arti...,0,0,1,1000,936,1774424569
30,2026-03-24T13:23:09.838659+00:00,1802,Daddy Yankee,True,200,NaN,chartmetric_liveevents_2024_2025\raw_json\arti...,0,0,1,1000,935,1774424569
37,2026-03-24T13:23:31.854517+00:00,206557,BTS,True,200,NaN,chartmetric_liveevents_2024_2025\raw_json\arti...,0,0,1,1000,928,1774424569
41,2026-03-24T13:23:43.219496+00:00,1702,Sia,True,200,NaN,chartmetric_liveevents_2024_2025\raw_json\arti...,0,0,1,1000,924,1774424569
43,2026-03-24T13:23:48.423911+00:00,2619,Miley Cyrus,True,200,NaN,chartmetric_liveevents_2024_2025\raw_json\arti...,0,0,1,1000,922,1774424569
50,2026-03-24T13:24:12.475693+00:00,558681,Harry Styles,True,200,NaN,chartmetric_liveevents_2024_2025\raw_json\arti...,0,0,1,1000,915,1774424569


In [131]:
print(df_faltantes[["ok", "status_code"]].value_counts(dropna=False))

ok    status_code
True  200            396
Name: count, dtype: int64


In [132]:
print("Distribución n_shows_2024 en faltantes:")
print(df_faltantes["n_shows_2024"].value_counts(dropna=False).sort_index())

print("\nDistribución n_shows_2025 en faltantes:")
print(df_faltantes["n_shows_2025"].value_counts(dropna=False).sort_index())

Distribución n_shows_2024 en faltantes:
n_shows_2024
0    396
Name: count, dtype: int64

Distribución n_shows_2025 en faltantes:
n_shows_2025
0    396
Name: count, dtype: int64


In [133]:
condicion_esperada = (
    (df_faltantes["ok"] == True) &
    (df_faltantes["status_code"] == 200) &
    (df_faltantes["n_shows_2024"] == 0) &
    (df_faltantes["n_shows_2025"] == 0)
)

print("Cantidad de faltantes que cumplen exactamente la condición esperada:", condicion_esperada.sum())
print("Total de faltantes:", len(df_faltantes))

Cantidad de faltantes que cumplen exactamente la condición esperada: 396
Total de faltantes: 396


In [134]:
df_casos_raros = df_faltantes[~condicion_esperada].copy()

print(df_casos_raros.shape)
df_casos_raros.head(20)

(0, 13)


,downloaded_at_utc,chartmetric_id,artist_name,ok,status_code,error_text,raw_json_path,n_shows_2024,n_shows_2025,n_requests_live_2024_2025,x_ratelimit_limit,x_ratelimit_remaining,x_ratelimit_reset


# AGREGO BAND al flat.csv

In [135]:
import os
import json
import pandas as pd

JSON_DIR = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\chartmetric_metadata\raw_json"
CSV_PATH = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\chartmetric_metadata\metadata_flat.csv"

# ------------------------------------------
# 1) extraer id + band desde JSON
# ------------------------------------------
rows_band = []

json_files = [f for f in os.listdir(JSON_DIR) if f.lower().endswith(".json")]

for fname in json_files:
    fpath = os.path.join(JSON_DIR, fname)

    try:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)

        obj = data.get("obj", data)

        rows_band.append({
            "response_id": obj.get("id"),   # ← clave alineada con tu CSV
            "band": obj.get("band")
        })

    except Exception as e:
        print(f"Error leyendo {fname}: {e}")

df_band = pd.DataFrame(rows_band).drop_duplicates(subset=["response_id"])

# ------------------------------------------
# 2) leer metadata_flat
# ------------------------------------------
df_metadata = pd.read_csv(CSV_PATH)

# ------------------------------------------
# 3) merge limpio (sin tocar nada más)
# ------------------------------------------
df_metadata = df_metadata.drop(columns=["band"], errors="ignore")

df_metadata = df_metadata.merge(
    df_band,
    on="response_id",
    how="left"
)

# ------------------------------------------
# 4) guardar
# ------------------------------------------
df_metadata.to_csv(CSV_PATH, index=False)

print("metadata_flat.csv actualizado con la columna band")
print(df_metadata["band"].value_counts(dropna=False))

metadata_flat.csv actualizado con la columna band
band
False    1011
True      389
Name: count, dtype: int64


# AGREGO MAS DEL JSON

In [136]:
import os
import json
import pandas as pd

JSON_DIR = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\chartmetric_metadata\raw_json"
CSV_PATH = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\chartmetric_metadata\metadata_flat.csv"

# ------------------------------------------
# 1) extraer id + variables desde JSON
# ------------------------------------------
rows_stats = []

json_files = [f for f in os.listdir(JSON_DIR) if f.lower().endswith(".json")]

for fname in json_files:
    fpath = os.path.join(JSON_DIR, fname)

    try:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)

        obj = data.get("obj", data)
        cm_stats = obj.get("cm_statistics", {})

        rows_stats.append({
            "response_id": obj.get("id"),

            "num_sp_editorial_playlists": cm_stats.get("num_sp_editorial_playlists"),
            "num_sp_playlists": cm_stats.get("num_sp_playlists"),
            "sp_editorial_playlist_total_reach": cm_stats.get("sp_editorial_playlist_total_reach"),
            "num_am_editorial_playlists": cm_stats.get("num_am_editorial_playlists"),
            "num_am_playlists": cm_stats.get("num_am_playlists"),
            "num_de_editorial_playlists": cm_stats.get("num_de_editorial_playlists"),
            "num_de_playlists": cm_stats.get("num_de_playlists"),
            "de_playlist_total_reach": cm_stats.get("de_playlist_total_reach"),
            "de_editorial_playlist_total_reach": cm_stats.get("de_editorial_playlist_total_reach"),
            "num_az_editorial_playlists": cm_stats.get("num_az_editorial_playlists"),
            "num_az_playlists": cm_stats.get("num_az_playlists"),
            "num_yt_editorial_playlists": cm_stats.get("num_yt_editorial_playlists"),
            "num_yt_playlists": cm_stats.get("num_yt_playlists"),
            "yt_playlist_total_reach": cm_stats.get("yt_playlist_total_reach"),

            "tiktok_top_video_views": cm_stats.get("tiktok_top_video_views"),
            "tiktok_top_video_comments": cm_stats.get("tiktok_top_video_comments"),
            "tiktok_track_posts": cm_stats.get("tiktok_track_posts")
        })

    except Exception as e:
        print(f"Error leyendo {fname}: {e}")

df_stats = pd.DataFrame(rows_stats).drop_duplicates(subset=["response_id"])

# ------------------------------------------
# 2) leer metadata_flat
# ------------------------------------------
df_metadata = pd.read_csv(CSV_PATH)

# ------------------------------------------
# 3) merge limpio
# ------------------------------------------
cols_to_replace = [
    "num_sp_editorial_playlists",
    "num_sp_playlists",
    "sp_editorial_playlist_total_reach",
    "num_am_editorial_playlists",
    "num_am_playlists",
    "num_de_editorial_playlists",
    "num_de_playlists",
    "de_playlist_total_reach",
    "de_editorial_playlist_total_reach",
    "num_az_editorial_playlists",
    "num_az_playlists",
    "num_yt_editorial_playlists",
    "num_yt_playlists",
    "yt_playlist_total_reach",
    "tiktok_top_video_views",
    "tiktok_top_video_comments",
    "tiktok_track_posts"
]

df_metadata = df_metadata.drop(columns=cols_to_replace, errors="ignore")

df_metadata = df_metadata.merge(
    df_stats,
    on="response_id",
    how="left"
)

# ------------------------------------------
# 4) guardar
# ------------------------------------------
df_metadata.to_csv(CSV_PATH, index=False)

print("metadata_flat.csv actualizado con variables de playlists y TikTok")

for col in cols_to_replace:
    print(f"\n{col}")
    print(df_metadata[col].notna().sum(), "valores no nulos")

metadata_flat.csv actualizado con variables de playlists y TikTok

num_sp_editorial_playlists
1398 valores no nulos

num_sp_playlists
1398 valores no nulos

sp_editorial_playlist_total_reach
1398 valores no nulos

num_am_editorial_playlists
1384 valores no nulos

num_am_playlists
1384 valores no nulos

num_de_editorial_playlists
1368 valores no nulos

num_de_playlists
1368 valores no nulos

de_playlist_total_reach
1368 valores no nulos

de_editorial_playlist_total_reach
1368 valores no nulos

num_az_editorial_playlists
1350 valores no nulos

num_az_playlists
1350 valores no nulos

num_yt_editorial_playlists
1373 valores no nulos

num_yt_playlists
1373 valores no nulos

yt_playlist_total_reach
1369 valores no nulos

tiktok_top_video_views
1394 valores no nulos

tiktok_top_video_comments
1394 valores no nulos

tiktok_track_posts
1394 valores no nulos


# MERGE metadata

In [137]:
ruta_metadata = r"chartmetric_metadata\metadata_flat.csv"

df_metadata = pd.read_csv(ruta_metadata)

print(df_metadata.shape)
df_metadata.head()

(1400, 51)


,downloaded_at_utc,requested_chartmetric_id,requested_artist_name,response_id,name,code2,pronoun_title,gender_title,record_label,hometown_city,...,de_playlist_total_reach,de_editorial_playlist_total_reach,num_az_editorial_playlists,num_az_playlists,num_yt_editorial_playlists,num_yt_playlists,yt_playlist_total_reach,tiktok_top_video_views,tiktok_top_video_comments,tiktok_track_posts
0,2026-03-23T14:29:43.317838,214945,Bad Bunny,214945,Bad Bunny,PR,he/him,male,Rimas Entertainment,Vega Baja,...,75271226.0,43152895.0,1098.0,1132.0,138.0,749.0,6.463153e+09,1.252662e+11,88400519.0,62886375.0
1,2026-03-23T14:29:45.564385,2762,Taylor Swift,2762,Taylor Swift,US,she/her,female,UMG,Reading,...,63668913.0,35123672.0,1570.0,1680.0,200.0,1014.0,1.444689e+10,1.308229e+11,107184246.0,59337202.0
2,2026-03-23T14:29:47.942302,3501,Bruno Mars,3501,Bruno Mars,US,he/him,male,WMG,Los Angeles,...,62197788.0,31743007.0,1025.0,1104.0,150.0,1082.0,1.267911e+10,7.282031e+10,58774870.0,46772998.0
3,2026-03-23T14:29:50.168041,3479,Justin Bieber,3479,Justin Bieber,CA,he/him,male,UMG,Stratford,...,22297011.0,16285892.0,1388.0,1440.0,152.0,1084.0,1.124400e+10,9.640824e+10,93459076.0,79842774.0
4,2026-03-23T14:29:52.750364,3852,The Weeknd,3852,The Weeknd,CA,he/him,male,UMG,Toronto,...,31880622.0,22194841.0,1186.0,1258.0,136.0,1134.0,8.969425e+09,4.772666e+10,39861263.0,19846655.0


In [138]:
df_metadata = df_metadata.rename(columns={
    "response_id": "chartmetric_id"
})

In [139]:
print("Duplicados en metadata:", df_metadata["chartmetric_id"].duplicated().sum())

Duplicados en metadata: 0


In [140]:
print(df_metadata.columns.tolist())

['downloaded_at_utc', 'requested_chartmetric_id', 'requested_artist_name', 'chartmetric_id', 'name', 'code2', 'pronoun_title', 'gender_title', 'record_label', 'hometown_city', 'cm_artist_rank', 'cm_artist_score', 'career_stage', 'career_stage_score', 'career_trend', 'career_trend_score', 'primary_genre', 'sp_followers', 'sp_monthly_listeners', 'sp_popularity', 'sp_playlist_total_reach', 'ins_followers', 'twitter_followers', 'tiktok_followers', 'tiktok_likes', 'ycs_subscribers', 'ycs_views', 'youtube_daily_video_views', 'youtube_monthly_video_views', 'deezer_fans', 'shazam_count', 'pandora_lifetime_streams', 'pandora_lifetime_stations_added', 'band', 'num_sp_editorial_playlists', 'num_sp_playlists', 'sp_editorial_playlist_total_reach', 'num_am_editorial_playlists', 'num_am_playlists', 'num_de_editorial_playlists', 'num_de_playlists', 'de_playlist_total_reach', 'de_editorial_playlist_total_reach', 'num_az_editorial_playlists', 'num_az_playlists', 'num_yt_editorial_playlists', 'num_yt_pla

In [141]:
df_metadata = df_metadata.rename(columns={
    "name": "artist_name"
})

In [142]:
df_final = df_metadata.merge(
    df_live_agg,
    on="chartmetric_id",
    how="left",
    suffixes=("_meta", "_live")
)

In [143]:
[col for col in df_final.columns if "name" in col.lower()]

['requested_artist_name', 'artist_name_meta', 'artist_name_live']

In [144]:
df_final[[
    "chartmetric_id",
    "requested_artist_name",
    "artist_name_meta",
    "artist_name_live"
]].head(15)

,chartmetric_id,requested_artist_name,artist_name_meta,artist_name_live
0,214945,Bad Bunny,Bad Bunny,Bad Bunny
1,2762,Taylor Swift,Taylor Swift,Taylor Swift
2,3501,Bruno Mars,Bruno Mars,Bruno Mars
3,3479,Justin Bieber,Justin Bieber,NaN
4,3852,The Weeknd,The Weeknd,The Weeknd
5,3380,Drake,Drake,Drake
6,2316,Rihanna,Rihanna,NaN
7,3648,Ed Sheeran,Ed Sheeran,Ed Sheeran
8,5596,Billie Eilish,Billie Eilish,Billie Eilish
9,3188,Lady Gaga,Lady Gaga,Lady Gaga


In [145]:
df_final["validacion_requested_vs_meta"] = (
    df_final["requested_artist_name"] == df_final["artist_name_meta"]
)

df_final["validacion_meta_vs_live"] = (
    df_final["artist_name_live"].isna() |
    (df_final["artist_name_meta"] == df_final["artist_name_live"])
)

In [146]:
print("Filas metadata:", df_metadata.shape[0])
print("Filas final:", df_final.shape[0])

Filas metadata: 1400
Filas final: 1400


In [147]:
print("Filas metadata:", df_metadata.shape[0])
print("Filas final:", df_final.shape[0])

Filas metadata: 1400
Filas final: 1400


In [148]:
print("Sin match en live:", df_final["artist_name_live"].isna().sum())

Sin match en live: 396


In [149]:
df_final.loc[df_final["artist_name_live"].isna(), ["chartmetric_id", "artist_name_meta"]].head(10)

,chartmetric_id,artist_name_meta
3,3479,Justin Bieber
6,2316,Rihanna
10,3963,Ariana Grande
26,178,Michael Jackson
29,3168,Selena Gomez
30,1802,Daddy Yankee
37,206557,BTS
41,1702,Sia
43,2619,Miley Cyrus
50,558681,Harry Styles


In [150]:
print("Duplicados en df_final por chartmetric_id:", df_final["chartmetric_id"].duplicated().sum())

Duplicados en df_final por chartmetric_id: 0


In [151]:
print("Filas metadata:", df_metadata.shape[0])
print("Filas final:", df_final.shape[0])
print("Duplicados en df_final por chartmetric_id:", df_final["chartmetric_id"].duplicated().sum())
print("Sin match en live:", df_final["artist_name_live"].isna().sum())

Filas metadata: 1400
Filas final: 1400
Duplicados en df_final por chartmetric_id: 0
Sin match en live: 396


# DF final

In [152]:
df_final.shape

(1400, 83)

In [153]:
df_final["validacion_meta_vs_live"].value_counts()

validacion_meta_vs_live
True     1399
False       1
Name: count, dtype: int64

In [154]:
df_final.loc[
    df_final["validacion_meta_vs_live"] == False,
    ["chartmetric_id", "artist_name_meta", "artist_name_live"]
]

,chartmetric_id,artist_name_meta,artist_name_live
1072,209368,SOFI TUKKER,Sofi Tukker


In [155]:
df_final.loc[
    df_final["chartmetric_id"] == 209368,
    ["artist_name_meta", "artist_name_live"]
] = ["Sofi Tukker", "Sofi Tukker"]

In [156]:
df_final.loc[df_final["chartmetric_id"] == 209368,
             ["artist_name_meta", "artist_name_live"]]

,artist_name_meta,artist_name_live
1072,Sofi Tukker,Sofi Tukker


In [157]:
df_final.columns

Index(['downloaded_at_utc', 'requested_chartmetric_id',
       'requested_artist_name', 'chartmetric_id', 'artist_name_meta', 'code2',
       'pronoun_title', 'gender_title', 'record_label', 'hometown_city',
       'cm_artist_rank', 'cm_artist_score', 'career_stage',
       'career_stage_score', 'career_trend', 'career_trend_score',
       'primary_genre', 'sp_followers', 'sp_monthly_listeners',
       'sp_popularity', 'sp_playlist_total_reach', 'ins_followers',
       'twitter_followers', 'tiktok_followers', 'tiktok_likes',
       'ycs_subscribers', 'ycs_views', 'youtube_daily_video_views',
       'youtube_monthly_video_views', 'deezer_fans', 'shazam_count',
       'pandora_lifetime_streams', 'pandora_lifetime_stations_added', 'band',
       'num_sp_editorial_playlists', 'num_sp_playlists',
       'sp_editorial_playlist_total_reach', 'num_am_editorial_playlists',
       'num_am_playlists', 'num_de_editorial_playlists', 'num_de_playlists',
       'de_playlist_total_reach', 'de_editoria

## elimino columnas innecesarias

In [158]:
cols_drop = [
    "downloaded_at_utc",
    "requested_chartmetric_id",
    "requested_artist_name",
    "artist_name_live",
    "validacion_requested_vs_meta",
    "validacion_meta_vs_live"
]

df_final = df_final.drop(columns=cols_drop)

In [159]:
df_final = df_final.rename(columns={
    "artist_name_meta": "artist_name",
    "code2": "country"
})

In [160]:
print(df_final.shape)
df_final.head()

(1400, 77)


,chartmetric_id,artist_name,country,pronoun_title,gender_title,record_label,hometown_city,cm_artist_rank,cm_artist_score,career_stage,...,main_currency_2025,shows_per_country_2025,shows_per_country_2024,n_shows_total,n_shows_with_capacity_total,total_capacity_total,avg_venue_capacity_total,n_cities_total,n_countries_total,shows_per_country_total
0,214945,Bad Bunny,PR,he/him,male,Rimas Entertainment,Vega Baja,4,98.976377,superstar,...,USD,9.0,NaN,45.0,44.0,1269124.0,28843.727273,5.0,5.0,9.000000
1,2762,Taylor Swift,US,she/her,female,UMG,Reading,1,99.387117,superstar,...,NaN,NaN,4.9,49.0,49.0,3185238.0,65004.857143,15.0,10.0,4.900000
2,3501,Bruno Mars,US,he/him,male,WMG,Los Angeles,2,99.207939,superstar,...,USD,16.0,6.5,55.0,51.0,1370187.0,26866.411765,11.0,6.0,9.166667
3,3479,Justin Bieber,CA,he/him,male,UMG,Stratford,3,99.079736,superstar,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3852,The Weeknd,CA,he/him,male,UMG,Toronto,7,98.740889,superstar,...,USD,22.0,2.5,49.0,47.0,3080179.0,65535.723404,27.0,4.0,12.250000


In [161]:
df_final.columns

Index(['chartmetric_id', 'artist_name', 'country', 'pronoun_title',
       'gender_title', 'record_label', 'hometown_city', 'cm_artist_rank',
       'cm_artist_score', 'career_stage', 'career_stage_score', 'career_trend',
       'career_trend_score', 'primary_genre', 'sp_followers',
       'sp_monthly_listeners', 'sp_popularity', 'sp_playlist_total_reach',
       'ins_followers', 'twitter_followers', 'tiktok_followers',
       'tiktok_likes', 'ycs_subscribers', 'ycs_views',
       'youtube_daily_video_views', 'youtube_monthly_video_views',
       'deezer_fans', 'shazam_count', 'pandora_lifetime_streams',
       'pandora_lifetime_stations_added', 'band', 'num_sp_editorial_playlists',
       'num_sp_playlists', 'sp_editorial_playlist_total_reach',
       'num_am_editorial_playlists', 'num_am_playlists',
       'num_de_editorial_playlists', 'num_de_playlists',
       'de_playlist_total_reach', 'de_editorial_playlist_total_reach',
       'num_az_editorial_playlists', 'num_az_playlists',
  

# descargo cohorte 1

In [162]:
df_final.to_csv("cohorte1_raw_1400.csv", index=False)